# Python prediction API demo
To run this notebook please install via:

In [ ]:
!uv sync --dev

## Load model
Here you just simply load the `Predictor` class and initialize without any arguments, it will automatically download and cache the weights *(this can be changed with arguments, but the defaults are recommended)*, load the model and prepare the necessary readers, preprocessors etc.

In [ ]:
from mini_trainer.deploy import Predictor

model = Predictor()

## Usage
### Prediction
The `Predictor` is incredibly simple to use; just call the model on a `list`/`tuple` of paths (`str`) or preloaded images (`torch.Tensor`/`np.ndarray`, or prebatched).

***Note:** most models expect preloaded images to be pre-resized to a specific input size, and provided in uint8 - handling of these cases may become more seamless in the future.*

In [ ]:
import numpy as np
import torch
from PIL import Image

images = [
    "examples/global_lepi/images/1730446/066d9c42ebd3c5b19f42de576ff82f83867f9b16.jpg",
    "examples/global_lepi/images/1956897/4ae9a90723346aeaa8ae76712d1762c8d2847632.jpg"
]

# With one image
pred1 = model(images[0])
print(pred1)

img0_pil = Image.open(images[0]).resize((512, 512)) # Currently the size is only handled automatically for string inputs

# NumPy
img0_np = np.transpose(np.asarray(img0_pil), (2, 1, 0)).copy() # Make sure you use C-H-W, not H-W-C
print(model(img0_np))
# PyTorch
img0_tensor = torch.from_numpy(img0_np).clone()
print(model(img0_tensor))

# With multiple images
pred2 = model(images)
print(pred2)

### Results
The returned object will always inherit from `mini_trainer.classifier.Prediction`, but may be a different subclass depending on the model (in particular `mini_trainer.hierarchical.model.HierarchicalPrediction`). 

#### Wrangling
This is extremely useful for quickly checking the results:

In [ ]:
import os

import numpy as np

from mini_trainer.utils.io import is_image

dir = "examples/global_lepi/images/1732697"
images = [p for n in os.listdir(dir) if is_image(p := os.path.join(dir, n))]

pred = model(images)
print(str(pred)[:250] + "...")

correct = np.array(pred.labels)[:, 0, 0] == os.path.basename(dir)
print(f'Accuracy: {np.mean(correct).item():.1%} ({int(np.sum(correct).item())}/{len(images)})')

#### Serialization/Storage
Or serialization via dict-JSON:

In [ ]:
import json
import tempfile

# Convert to dict
print("Dictionary serialization:")
print(*pred.to_dict()[:3], "", sep="\n")

# Or save to disk (powered directly by json.dumps and pred.to_dict)
with tempfile.NamedTemporaryFile(suffix=".json") as file:
    pred.save(file.name)
    with open(file.name) as f:
        print(
            "Contents of stored predictions:\n",
            "".join(f.readlines())[:250] + "..."
        )
    with open(file.name) as f:
        reconstructed = json.load(f)


def is_equal(a, b):
    if hasattr(a, "__iter__") and len(a) > 1 and hasattr(b, "__iter__") and len(b) > 1:
        return all(map(lambda xy : is_equal(*xy), zip(a, b)))
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        min_abs = max(1e-6, min(map(abs, [a, b])))
        diff = abs(a - b)
        rel_diff = diff / min_abs
        if rel_diff < 1e-2:
            return True
    return a == b


def check_equal(d1_d2 : tuple[dict, dict]):
    d1, d2 = d1_d2
    if d1.keys() != d2.keys():
        return False
    keys = list(d1.keys())
    return all([is_equal(d1[k], d2[k]) for k in keys])


good_reconstruction = all(list(map(check_equal, zip(pred.to_dict(), reconstructed["results"]))))
print(f'Reconstruction {"succeeded" if good_reconstruction else "failed"}!')